# 💳 Credit Card Fraud Financial Loss Prediction

**Section:** 402 B  
**Problem Type:** Regression  
**Models:** Linear Regression & Random Forest Regressor  
**Metrics:** MAE & RMSE  

### Project objective
Estimate the **financial loss/exposure associated with a transaction being fraudulent**.

### Important target definition
The dataset contains transaction amount (`amt`) and fraud label (`is_fraud`), but it does not contain a separate confirmed loss column.

We therefore create a transparent **realized fraud-loss proxy**:

`financial_loss = amt × is_fraud`

So:
- a non-fraudulent transaction has loss = 0
- a fraudulent transaction has loss = its transaction amount

The model is trained on the full dataset and uses the **transaction amount as an input feature**, along with transaction/time/merchant-related features, to estimate the expected financial exposure.

This matches the assignment's regression framing more closely than predicting `amt` only among already-known fraudulent transactions.


In [ ]:
# Install the exact scikit-learn version used by the project
!pip -q install scikit-learn==1.6.1


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

print("Libraries imported successfully!")


## 1. Upload the Kaggle dataset

Upload the same credit-card transaction CSV/ZIP used previously.


In [ ]:
uploaded = files.upload()

import os, zipfile

uploaded_name = next(iter(uploaded.keys()))

if uploaded_name.lower().endswith(".zip"):
    with zipfile.ZipFile(uploaded_name, "r") as zip_ref:
        zip_ref.extractall("dataset")

    csv_files = []
    for root, dirs, files_in_dir in os.walk("dataset"):
        for f in files_in_dir:
            if f.lower().endswith(".csv"):
                csv_files.append(os.path.join(root, f))

    if not csv_files:
        raise FileNotFoundError("No CSV file was found inside the ZIP file.")

    csv_path = csv_files[0]
else:
    csv_path = uploaded_name

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
# Basic dataset check
print("Fraud counts:")
print(df["is_fraud"].value_counts())

print("\nMissing values in selected columns:")
selected_check = [
    "trans_date_trans_time", "amt", "category", "gender",
    "city", "state", "job", "lat", "long", "city_pop",
    "merch_lat", "merch_long", "is_fraud"
]
print(df[selected_check].isnull().sum())


## 2. Create the regression target

The dataset has no separate confirmed financial-loss column.

For this project, we use:

**financial_loss = transaction_amount × fraud_label**

This gives a simple, explainable proxy for the amount financially exposed to a fraudulent transaction.


In [ ]:
# Create the financial-loss target
df["financial_loss"] = df["amt"] * df["is_fraud"]

print("Financial loss summary:")
print(df["financial_loss"].describe())

print("\nTotal historical fraud-loss proxy:", df["financial_loss"].sum())
print("Number of fraudulent transactions:", int(df["is_fraud"].sum()))


## 3. Feature engineering

We extract hour, day, month and day-of-week from the transaction timestamp.

The transaction amount is intentionally kept as an **input feature**, as required by the project brief.


In [ ]:
# Convert timestamp and extract time features
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])

df["hour"] = df["trans_date_trans_time"].dt.hour
df["day"] = df["trans_date_trans_time"].dt.day
df["month"] = df["trans_date_trans_time"].dt.month
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek

features = [
    "amt",
    "category",
    "gender",
    "city",
    "state",
    "job",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "hour",
    "day",
    "month",
    "day_of_week"
]

target = "financial_loss"

X = df[features].copy()
y = df[target].copy()

print("Input features:")
print(features)
print("\nTarget:", target)
print("\nX shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

categorical_features = ["category", "gender", "city", "state", "job"]
numeric_features = [
    "amt", "lat", "long", "city_pop",
    "merch_lat", "merch_long",
    "hour", "day", "month", "day_of_week"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 4. Linear Regression


In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))

print(f"Linear Regression MAE: {linear_mae:.2f}")
print(f"Linear Regression RMSE: {linear_rmse:.2f}")


## 5. Random Forest Regressor


In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"Random Forest MAE: {rf_mae:.2f}")
print(f"Random Forest RMSE: {rf_rmse:.2f}")


## 6. Compare the models


In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "MAE": [linear_mae, rf_mae],
    "RMSE": [linear_rmse, rf_rmse]
})

display(results)

plt.figure(figsize=(8, 5))
x = np.arange(len(results))
width = 0.35

plt.bar(x - width/2, results["MAE"], width, label="MAE")
plt.bar(x + width/2, results["RMSE"], width, label="RMSE")

plt.xticks(x, results["Model"])
plt.ylabel("Error")
plt.title("Model Performance Comparison")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Save the better model

We select the model with the lower RMSE on the held-out test set.


In [ ]:
if rf_rmse <= linear_rmse:
    best_model = rf_model
    best_model_name = "Random Forest Regressor"
else:
    best_model = linear_model
    best_model_name = "Linear Regression"

print("Selected model:", best_model_name)

joblib.dump(best_model, "fraud_financial_loss_model.pkl")
print("Saved as: fraud_financial_loss_model.pkl")


## 8. Download the trained model

Run this cell and download the `.pkl` file. We will use it in the Streamlit website.


In [ ]:
files.download("fraud_financial_loss_model.pkl")


## 9. Viva explanation

**What is the target?**  
Financial loss/exposure associated with a fraudulent transaction.

**How was the target created?**  
`financial_loss = amt × is_fraud`.

**Why is amount an input?**  
A larger transaction amount can create larger potential financial exposure, so the transaction amount is a valid predictor.

**Why regression?**  
The output is a continuous monetary value rather than only a fraud/not-fraud class.

**Why Random Forest?**  
We compare Linear Regression and Random Forest using MAE and RMSE. The model with lower test error is selected.

**Important limitation:**  
The Kaggle dataset does not provide a separately verified post-fraud loss amount. Therefore, `financial_loss` is a transparent proxy based on transaction amount and the dataset's fraud label; it should not be described as a confirmed bank loss.
